# 01b — Reproducible Discovery

## The descriptive question

This notebook asks one question of the recorded data:

> In the UCI *Wholesale customers* data set, what is the difference in median annual grocery spending between clients classified as Retail and Horeca?

The question names an outcome (`Grocery`), an existing group (`Channel`), and a descriptive statistic: the Retail median minus the Horeca median. Before calculating it, identify what each row represents and inspect the distribution that the statistic will summarize.

### Learning goals

By the end of this notebook, you should be able to:

- identify the observational unit, data source, variable roles, and limits of a
  data set;
- use basic pandas selection, grouping, and plotting to make a descriptive
  comparison; and
- use a Welch two-sample t-test for a mean difference, a seeded permutation test for a median difference, and a bounded claim with its next evidence question.

## Frame the comparison

Four ideas guide this comparison:

- **Observational unit:** what one row represents.
- **Population or process:** what broader set or process the records might, or
  might not, stand for.
- **Variable role:** whether a column supplies a group, outcome, identifier, or
  contextual detail for this question.
- **Provenance:** where the file came from and what its documentation does and
  does not tell us.

### Data and scope

The UCI *Wholesale customers* data set contains annual spending records for 440 clients of a wholesale distributor. Its `Channel` field records a Horeca or Retail classification.

- one row is one recorded client;
- `Channel` is the **existing group variable**;
- `Grocery` is the annual-spending **outcome** we will compare; and
- `Region` supplies context for the comparison.

`Channel` records an existing classification. We use it to compare groups; we do not infer it from the spending values. The comparison describes the recorded clients. It does not establish that channel caused spending or that the same difference applies to other wholesalers or a later period.

## Load the source

Data source: Cardoso, M. (2013). *Wholesale customers* [Dataset]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5030X>. The source archive is available under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

The spending values use source monetary units. They are not dollars. The documentation does not identify the sampling method, time period, or business decision that spending should inform.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import permutation_test, ttest_ind

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/292/wholesale%2Bcustomers.zip"

customers = pd.read_csv(DATA_URL, compression="zip")
customers.shape

In [ ]:
customers.head()

### Review pandas basics

Review the accompanying `01b-data-work-bridge.ipynb` before continuing if you want practice selecting columns, filtering rows, grouping, or making a simple plot.

## Inspect the records and variables

The source encodes `Channel` and `Region` as numbers. Map `Channel` to a readable label before you group the records.

In [ ]:
channel_labels = {1: "Horeca", 2: "Retail"}

customers["channel"] = customers["Channel"].map(channel_labels)

customers[["Channel", "channel", "Region", "Grocery"]].head()

In [ ]:
customers[["Channel", "Region", "Grocery"]].isna().sum()

In [ ]:
customers["channel"].value_counts().sort_index()

These fields have no missing values. Both group summaries use the observed `Grocery` values. The check does not describe the sampling method or the time period.

### Pause: separate calculation from generalization

Does this file support calculating the stated median difference? Cite evidence from the checks above.

Then name one broader claim that the calculation still cannot support.

##### Answer

Yes. `Channel` and `Grocery` have no missing values in the 440 recorded rows, so the file supports calculating the difference in medians for these clients. The calculation cannot show that the same difference applies to all wholesalers or a later period because the documentation does not describe the sampling method or time period.

## Summarize the groups

The question calls for the difference in group medians. This summary shows each group's count, mean, and median grocery spending. The counts show how many records contribute to the comparison.

In [ ]:
group_summary = (
    customers.groupby("channel")["Grocery"].agg(["count", "mean", "median"]).sort_index()
)

group_summary

The gap between the mean and median suggests a right tail. The **difference in group medians** is less pulled by a few large values. The next plot lets us inspect the distribution behind that choice. Another question or distribution might call for a different statistic.

In [ ]:
median_difference = group_summary.loc["Retail", "median"] - group_summary.loc["Horeca", "median"]

print(f"Retail minus Horeca median grocery spending: {median_difference:,.0f}")

### Inspect the distributions

The table supplies the declared statistic. The box plot shows the groups' centers, spread, and unusually large observations in one view.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

customers.boxplot(column="Grocery", by="channel", ax=ax)
ax.set_title("Annual grocery spending by existing channel")
ax.set_xlabel("Existing channel")
ax.set_ylabel("Annual grocery spending (source monetary units)")
plt.suptitle("")
plt.show()

### Pause: state the descriptive result

Write a descriptive claim from the table and plot.

> In these 440 recorded clients, the typical grocery-spending value is [higher/lower] for [channel] than for [channel], by about [amount] source monetary units.

Then name one feature of the plot that the median difference does not show.

##### Answer

In these 440 recorded clients, the Retail median is **9,706 source monetary units** higher than the Horeca median (12,390 versus 2,684). The box plot shows substantial variation in both groups. It also shows several very large Retail values. The median difference summarizes the groups. It does not describe every client or explain why the channels differ.

## A familiar test of mean difference

Start with a familiar statistical tool. A **Welch two-sample t-test** compares the means of two independent groups. Unlike the pooled two-sample t-test, it does not require the two groups to have equal variances.

For these records, it asks whether the Retail and Horeca clients have different **mean** annual grocery spending. This is a useful SciPy refresher, but it is not the same as the notebook's original question about the median difference.

In [ ]:
retail_grocery = customers.loc[customers["channel"] == "Retail", "Grocery"]
horeca_grocery = customers.loc[customers["channel"] == "Horeca", "Grocery"]

mean_difference = retail_grocery.mean() - horeca_grocery.mean()
welch_result = ttest_ind(retail_grocery, horeca_grocery, equal_var=False)

print(f"Retail minus Horeca mean grocery spending: {mean_difference:,.0f}")
print(f"Welch t statistic: {welch_result.statistic:.2f}")
print(f"p-value: {welch_result.pvalue:.3g}")

The mean difference is 12,361 source monetary units, and the p-value is well below 0.001. Under the test's no-mean-difference model, a difference this large would be very unusual. Thus, this test provides very strong evidence that mean annual grocery spending differs between the Retail and Horeca groups represented by these records.

The result is limited in three ways:

1. It tests a difference in means, while our original descriptive question uses medians because the distribution is right-skewed.
2. The test treats the records within each channel as independent draws from an underlying channel distribution. This i.i.d.-within-group assumption, together with a sampling process that represents the groups of interest, is what permits an inference beyond the 440 records. The source does not document that sampling process. This limits how confidently we can generalize this finding.
3. The test also does not show that channel caused spending to differ or that the difference matters for a business decision.

## Test the original median difference

The Welch test is a useful check on group means, but it does not answer the original median-difference question. For that question, ask: is the observed median difference statistically significant, or could it be a coincidence of the recorded channel labels?

To test this, ask: if `Channel` had no relationship with `Grocery` spending in these records, how often would randomly reassigning the same 142 Retail and 298 Horeca labels produce a median difference at least as large as 9,706 source monetary units?

The labels are real business classifications. Here, “coincidence” refers only to the observed pairing of those labels with the grocery values under the no-relationship model.

Work through that question before using statistical terms:

1. Assume that the existing `Channel` classification has no relationship with annual `Grocery` spending in these recorded clients.
2. Keep the same 440 grocery-spending values and the same group sizes: 142 Retail clients and 298 Horeca clients.
3. Randomly reassign the existing channel labels to those values 5,000 times. Each reassignment creates a new pair of groups with the same number of Retail and Horeca labels, but does not preserve the original pairing of a client's label and spending value.
4. Calculate the median difference after each reassignment. The 5,000 differences vary because each reassignment places different grocery values in the two groups. If channel has no relationship with grocery spending, these differences show the range that can arise from grouping these same values into groups of these sizes.
5. Compare the observed difference, 9,706 source monetary units, with the 5,000 shuffled differences.

SciPy provides `scipy.stats.permutation_test` for this repeated-reassignment calculation. We provide the two observed groups, the median-difference statistic, and the number of reassignments. SciPy performs the shuffles and returns the observed statistic and p-value.

A permutation test is a **resampling method**. It repeatedly rearranges the observed channel labels without replacement to create a reference distribution under the no-relationship model.

The code below carries out that comparison. A seed sets the starting state of the pseudorandom generator. Reusing that seed lets another reader reproduce this simulated result. It does not make the conclusion valid by itself.

In [ ]:
def median_difference(x, y):
    return np.median(x) - np.median(y)


rng = np.random.default_rng(7130)

permutation_result = permutation_test(
    (retail_grocery, horeca_grocery),
    statistic=median_difference,
    permutation_type="independent",
    alternative="two-sided",
    n_resamples=5_000,
    rng=rng,
)

print(f"Observed median difference: {permutation_result.statistic:,.0f}")
print(f"Simulated p-value: {permutation_result.pvalue:.4f}")

### Pause: choose the claim

The notebook now has a Welch result for the mean difference and a permutation result for the median difference. Which result directly answers the original question? Draft a bounded conclusion from that result, then name one causal claim it cannot support.

##### Answer

The permutation result directly answers the original median-difference question. The bounded conclusion is that `Channel` is associated with `Grocery` spending in these recorded clients. The result does not show that changing a client's channel would cause its grocery spending to change.

The observed difference is 9,706 source monetary units. Only a very small share of the 5,000 shuffled reassignments produced a difference at least that far from zero. That makes the no-relationship explanation difficult to reconcile with these recorded clients.

### In formal terms

The **null hypothesis** says that annual `Grocery` spending has no relationship with the existing `Channel` labels in these records. Under that hypothesis, the original pairing of labels and grocery values carries no information about spending, so reassigning the fixed number of labels produces the reference differences described above.

The collection of shuffled median differences is the **permutation distribution**, also called a **null distribution**. The **p-value** is the share of simulated differences at least as far from zero as the observed difference. Here, the simulated p-value is 0.0004.

**Bounded conclusion:** In these recorded clients, the existing `Channel` classification is associated with annual `Grocery` spending. The observed median difference is unlikely under the stated no-relationship model.

**Not a causal conclusion:** This test does not show that changing a client's channel would cause its grocery spending to change. The data do not describe an intervention, the sampling process, or other differences between Retail and Horeca clients.

## Record the analysis

Use this record to make the calculation inspectable. Another reader can see the data source, question, code, statistic, simulation settings, and conclusion. Reproducibility makes those choices visible. It does not show that the difference would recur in another sample or explain why it exists.

| Field | Record for this first analysis |
| --- | --- |
| Question | How does annual grocery spending differ between the recorded Retail and Horeca clients? |
| Observational unit | One recorded wholesale-distributor client. |
| Data source and scope | UCI Wholesale customers; 440 records; source monetary units; sampling period and frame not documented here. |
| Group and outcome | Existing `Channel` classification; annual `Grocery` spending. |
| Declared statistic | Retail median grocery spending minus Horeca median grocery spending. |
| Descriptive result | Retail's median annual grocery spending is 9,706 source monetary units higher than Horeca's (12,390 versus 2,684). |
| Familiar mean test | Welch two-sample t-test; Retail mean is 12,361 source monetary units higher; p-value well below 0.001. |
| Null-model preview | Two-sided permutation test of the median difference; 5,000 reassignments; seed 7130; simulated p-value 0.0004. |
| Supported claim | A descriptive difference in these recorded clients and evidence against this stated no-difference model; not a causal or population-general claim. |

## The course analytical framework

This is the course analytical framework. Each later notebook extends it with method-specific evidence.

```text
Question and scope
→ data source and observational unit
→ variable roles and representation
→ statistic or method
→ evidence
→ bounded claim
→ next evidence question or decision
```

## Next time

This notebook describes a difference in the recorded clients and previews one null-model result. The next two meetings build the evidence further:

- **02a — Permutation reasoning:** Why use a permutation test for the median difference? What does the tool preserve and break when it reassigns channel labels? How does that create the null distribution and p-value?
- **02b — Bootstrap reasoning:** How uncertain is this estimated median
  difference under a stated sampling-resampling assumption?

Both meetings use the same declared statistic and build on resampling. In 02a, permutation reasoning repeatedly reassigns the observed channel labels without replacement to explain the previewed result. In 02b, bootstrap reasoning repeatedly samples the observed clients with replacement to estimate the statistic's uncertainty under a stated resampling assumption.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the
[INSY 7130 course-materials README](../../README.md).